# This is the file responsible for producing datasets in different Indian Languages. 
We have used the English words provided in `~/btp1-2026/llm-latent-language/data/langs/en/clean.csv`
We are using the ai4bharat/indictrans2-en-indic-dist-200M model for translations.
The paper used DeepL translator which is really good at what it does, but the api is paid and it is a black-box.
ai4bharat is an open source alternative.

You can translate to your required languages by modifying the `target_languages` array. 
If you want to translate to say, x, y, z languages, just redeclare it with only your languages `target_languages = [x, y, z]`.

In [8]:
```
import torch
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

# Redirect tokenization_utils to expose PreTrainedTokenizerBase
import transformers.tokenization_utils
transformers.tokenization_utils.PreTrainedTokenizerBase = transformers.tokenization_utils_base.PreTrainedTokenizerBase

# Now import IndicTransToolkit and the rest
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

# 1. Configuration
MODEL_NAME = "ai4bharat/indictrans2-en-indic-dist-200M"
SRC_LANG = "eng_Latn"
HI_LANG = "hin_Deva"
target_languages = ["ben_Beng", "tam_Taml", "tel_Telu", "mar_Deva"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Using device: {DEVICE}")

# 2. Load Model, Tokenizer, and Processor
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, 
    trust_remote_code=True, 
    torch_dtype=DTYPE
).to(DEVICE)
model.eval()

ip = IndicProcessor(inference=True)

# 3. Read English Words
df = pd.read_csv("en_compressed.csv")
words = df["en_words"].dropna().astype(str).str.strip().tolist()

# 4. Translation Function
def translate_batch(text_list, target_lang, batch_size=64):
    translations = []
    for i in range(0, len(text_list), batch_size):
        chunk = text_list[i : i + batch_size]
        
        # Preprocess using IndicProcessor
        preprocessed = ip.preprocess_batch(chunk, src_lang=SRC_LANG, tgt_lang=target_lang)
        
        # Tokenize
        inputs = tokenizer(
            preprocessed,
            truncation=True,
            padding="longest",
            return_tensors="pt"
        ).to(DEVICE)
        
        # Generate tokens
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                num_beams=1,  # Greedy search for rapid single-word lexical lookup
                max_length=32
            )
            
        # Decode and postprocess
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        postprocessed = ip.postprocess_batch(decoded, lang=target_lang)
        translations.extend(postprocessed)
        
    return translations

# 5. Run Translations
print(f"Translating to Hindi ({HI_LANG})...")
hi_translations = translate_batch(words, HI_LANG)

for L1_LANG in target_languages:
    print(f"Translating to Language ({L1_LANG})...")
    l1_translations = translate_batch(words, L1_LANG)
    column_name = L1_LANG.split("_")[0] + "_word"
    # 6. Save formatted 3-column CSV
    out_df = pd.DataFrame({
        "en_word": words,
        "hi_word": hi_translations,
        column_name: l1_translations
    })
    output_file = f"data/{L1_LANG}_words.csv"
    out_df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"Done! Saved translated dataset to {output_file}")
    
print(f"DONE! Translation finished for all target languages.")
```

SyntaxError: invalid syntax (3298964547.py, line 1)

# Observations with above Code block:
1. We are getting a lot of words with punctuations added to their end.
2. Many words are translated not as nouns (even though a version exists for them as noun).
> For example,

> row 90 (delete) is हटाएँ in Hindi (a verb), rather it should be हटाना. In Bengali, it is মুছে ফেলুন (polite imperative: "please delete").

> row 109 (advance), it is आगे बढ़ें in Hindi (verb), which should be प्रगति (noun)

3. Many words are translated to Compound words.
> For example,

> row 11 (spring), is वसंत ऋतु in Hindi and Marathi, which should be just वसंत

> row 70 (thousand), is हज़ारों in Hindi, should be just हज़ार

> row 128 (left), is বাঁ দিকে ("towards left") in Bengali, should be just বাম or বাঁ

4. Several words were translated into oblique cases (locative, genitive, accusative) or plurals instead of standard nominative singular.
> In Marathi
*   `flower` translated as **`फुलांची`** (genitive: _"of the flowers"_). Change to **`फूल`**. 
*   `gold` translated as **`सोन्याचे`** (genitive: _"of gold"_). Change to **`सोने`**.
*   `ground` translated as **`जमिनी`** (inflected/plural). Change to **`जमीन`**.
*   `north` translated as **`उत्तरेला`** (_"to the north"_). Change to **`उत्तर`**.
*   `south` / `east` translated as **`दक्षिणेकडे`**, **`पूर्वेकडे`** (_"towards the south/east"_). Change to **`दक्षिण`**, **`पूर्व`**.
> In Telugu
*   `mouth` translated as **`నోటి`** (genitive: _"of the mouth"_). Change to **`నోరు`**.
*   `eye` translated as **`కంటి`** (genitive: _"of the eye"_). Change to **`కన్ను`**.
*   `foot` translated as **`పాదాలు`** (plural: _"feet"_). Change to **`పాదం`**. 
*   `north` translated as **`ఉత్తరాన`** (locative: _"in the north"_). Change to **`ఉత్తరం`**.




Isolated tokens like "tie", "mouth", "advance", or "flower" lack grammatical role markers. Because encoder-decoder models like IndicTrans2 are trained almost entirely on full sentences rather than lexical dictionary entries, translating an isolated lemma causes two failure modes (points 2 and 4 above):
1. Part-of-Speech (POS) Ambiguity: A homonym like "advance", "defeat", or "tie" can be interpreted as an imperative command (e.g., "Advance!" $\rightarrow$ "आगे बढ़ें", "ಮುందుకు సాగండి", or "Delete!" $\rightarrow$ "मुछे ফেলুন") rather than a noun.

2. Oblique / Inflected Case Suffixes: Without a head syntactic structure, the decoder frequently predicts locative, genitive, or plural inflections (e.g., Tamil "பையில்" = "in the bag", Marathi "फुलांची" = "of the flowers", Telugu "నోటి" = "of the mouth").


To solve these issues, we need to carry out the translation in a way, such that the model knows it is supposed to translate into a noun. That should solve the obliques cases issue, then we can work on the compound words. 
While going through the section `A.1 Word translation` in the paper "**Do Llamas Work in English?**",
They write 
`During these translations we translated both the individual words alongside their cloze sentences`
`For each word translation, we include the context of the cloze task to disambiguate homonyms.`
`Additionally, we parse through the translations and filter any cloze translations where the target word doesn’t align with the expected word from the individual word translation,`

We can conclude the following:
#### By translating words within a carrier frame, the decoder is syntactically constrained to generate a base nominative singular noun (lemma).

##### The Carrier Sentence Frame ("This is a {word}."):
Embedding the word into a simple sentence (e.g., "This is a {word}." $\rightarrow$ Hindi "यह एक {शब्द} है।") guarantees that the model treats {word} as the nominal predicate subject/object in the nominative case.

In [1]:
import torch
import pandas as pd
import re
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

# 1. Configuration
MODEL_NAME = "ai4bharat/indictrans2-en-indic-dist-200M"
SRC_LANG = "eng_Latn"
HI_LANG = "hin_Deva"
target_languages = ["ben_Beng", "tam_Taml", "tel_Telu", "mar_Deva"]


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

# 2. Load Model & Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, 
    trust_remote_code=True, 
    torch_dtype=DTYPE
).to(DEVICE)
model.eval()

ip = IndicProcessor(inference=True)

# 3. Load English Words
df = pd.read_csv("en_compressed.csv")
words = df["en_words"].dropna().astype(str).str.strip().tolist()

# 4. Method 1: Construct Contextual Carrier Sentences
# Using a copular frame: "This is a {word}."
carrier_sentences = [f"This is a {w}." for w in words]

def batch_translate(sentences, tgt_lang, batch_size=32):
    results = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        preprocessed = ip.preprocess_batch(batch, src_lang=SRC_LANG, tgt_lang=tgt_lang)
        
        inputs = tokenizer(
            preprocessed, 
            truncation=True, 
            padding="longest", 
            return_tensors="pt"
        ).to(DEVICE)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                num_beams=2,
                max_length=40
            )
            
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        postprocessed = ip.postprocess_batch(decoded, lang=tgt_lang)
        results.extend(postprocessed)
    return results

# 5. Extract Extracted Noun Lemma from Carrier Frame
def extract_lemma(translated_sentence, target_lang_code):
    """
    Extracts the translated noun by stripping boilerplate carrier phrasing
    (e.g., 'यह एक ... है।', 'हे एक ... आहे.', 'ఇది ఒక ...')
    """
    text = translated_sentence.strip()
    # Strip terminal punctuation/dandas
    text = re.sub(r'[।\.!?]', '', text).strip()
    
    # Strip boilerplate patterns by language
    if target_lang_code == 'hin_Deva':
        # "यह एक [noun] है"
        text = re.sub(r'^(यह\s+(एक\s+)?)', '', text)
        text = re.sub(r'\s+है$', '', text)
    elif target_lang_code == 'mar_Deva':
        # "हे एक [noun] आहे"
        text = re.sub(r'^(हे\s+(एक\s+)?)', '', text)
        text = re.sub(r'\s+आहे$', '', text)
    elif target_lang_code == 'ben_Beng':
        # "এটি একটি [noun]"
        text = re.sub(r'^(এটি\s+(একটি\s+)?)', '', text)
        text = re.sub(r'\s+(হয়|হলো)$', '', text)
    elif target_lang_code == 'tel_Telu':
        # "ఇది ఒక [noun]"
        text = re.sub(r'^(ఇది\s+(ఒక\s+)?)', '', text)
    elif target_lang_code == 'tam_Taml':
        # "இது ஒரு [noun]"
        text = re.sub(r'^(இது\s+(ஒரு\s+)?)', '', text)
        text = re.sub(r'\s+ஆகும்$', '', text)
        
    return text.strip()

# Translating Hindi
print(f"Translating framed sentences into Hindi ({HI_LANG})...")
hi_sentences = batch_translate(carrier_sentences, HI_LANG)
hi_lemmas = [extract_lemma(s, HI_LANG) for s in hi_sentences]

# 6. Fallback & Cross-Verification (As in Wendler et al.)
# Also translate the minimal "the {word}" variant to verify agreement

determiner_words = [f"the {w}" for w in words]
hi_dets = batch_translate(determiner_words, HI_LANG)
final_hi = []
for raw_w, lemma_h, det_h in zip(words, hi_lemmas, hi_dets):
    det_clean = re.sub(r'[।\.!?]', '', det_h).strip()
    # If carrier sentence extracted word is clean, use it; else fallback to det
    final_hi.append(lemma_h if (len(lemma_h) > 0 and len(lemma_h.split()) == 1) else det_clean)

   

for TARGET_L1 in target_languages:
    L1_NAME = TARGET_L1.split('_')[0]
    print(f"Translating framed sentences into Regional Language ({TARGET_L1})...")
    l1_sentences = batch_translate(carrier_sentences, TARGET_L1)
    l1_lemmas = [extract_lemma(s, TARGET_L1) for s in l1_sentences]
    l1_dets = batch_translate(determiner_words, TARGET_L1)
    
    final_l1 = []
    
    for raw_w, lemma_l, det_l in zip(words, l1_lemmas, l1_dets):
        det_clean = re.sub(r'[।\.!?]', '', det_l).strip()
        final_l1.append(lemma_l if (len(lemma_l) > 0 and len(lemma_l.split()) == 1) else det_clean)
    
    # 7. Output Final CSV
    out_df = pd.DataFrame({
        'en_word': words,
        'hi_word': final_hi,
        f'{L1_NAME}_word': final_l1
    })
    out_df.to_csv(f"data/method1/{L1_NAME}_Deva_words_method1.csv", index=False, encoding="utf-8-sig")
    print("Saved cleanly extracted nouns dataset!")

/raid/home/dgx1405/miniconda3/envs/indic_trans/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/raid/home/dgx1405/miniconda3/envs/indic_trans/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/raid/home/dgx1405/miniconda3/envs/indic_trans/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Translating framed sentences into Hindi (hin_Deva)...
Translating framed sentences into Regional Language (ben_Beng)...
Saved cleanly extracted nouns dataset!
Translating framed sentences into Regional Language (tam_Taml)...
Saved cleanly extracted nouns dataset!
Translating framed sentences into Regional Language (tel_Telu)...
Saved cleanly extracted nouns dataset!
Translating framed sentences into Regional Language (mar_Deva)...
Saved cleanly extracted nouns dataset!


In [14]:
import pandas as pd
import re

files = {
    'tam': 'data/tam_Taml_words.csv',
    'tel': 'data/tel_Telu_words.csv',
    'mar': 'data/mar_Deva_words.csv',
    'ben': 'data/ben_Beng_words.csv'
}

def clean(filepath, target_col):
    df = pd.read_csv(filepath)
    
    # 1. Strip trailing punctuation, dandas, and extra whitespace
    df['en_word'] = df['en_word'].astype(str).str.strip()
    df['hi_word'] = df['hi_word'].astype(str).str.replace(r'[।\.!?]', '', regex=True).str.strip()
    df[target_col] = df[target_col].astype(str).str.replace(r'[।\.!?]', '', regex=True).str.strip()
    
    # 2. Check for empty entries
    empty_mask = (df['en_word'] == '') | (df['hi_word'] == '') | (df[target_col] == '')
    if empty_mask.sum() > 0:
        print(f"Warning: {empty_mask.sum()} empty rows in {filepath}")
        
    # 3. Check for identical tokens across languages (Cognate check for hi vs mar)
    if 'mar' in target_col:
        cognates = df[df['hi_word'] == df[target_col]]
        print(f"Hindi-Marathi exact string overlaps: {len(cognates)} rows (e.g., {cognates['en_word'].head(3).tolist()})")
        
    # Save cleaned file
    clean_path = f"data/clean/{filepath.split('/')[1]}"
    df.to_csv(clean_path, index=False, encoding='utf-8-sig')
    print(f"Cleaned file written to {clean_path}")
    return df

cleaned_dfs = {}
for k, fname in files.items():
    cleaned_dfs[k] = clean(fname, f"{k}_word")

Cleaned file written to data/clean/tam_Taml_words.csv
Cleaned file written to data/clean/tel_Telu_words.csv
Hindi-Marathi exact string overlaps: 33 rows (e.g., ['forest', 'grid', 'river'])
Cleaned file written to data/clean/mar_Deva_words.csv
Cleaned file written to data/clean/ben_Beng_words.csv


Choosing the carrier frame as "This is a {word}", led to the translator model getting confused when words = "one", "two", "summer", "middle", etc.
So we are hard-coding those words.
For words like "left", it was interpreted as political "left", hence the word "वामपंथी" in Hindi and similar in other languages. 
Regex stripping on translations of "This is a one", "This is a two", etc led to only "है" in the final dataset. 

In [2]:
import pandas as pd
import re

# Comprehensive mapping of ground-truth corrections
CORRECTIONS = {
    'one': {'hi': 'एक', 'ben': 'এক', 'mar': 'एक', 'tel': 'ఒకటి', 'tam': 'ஒன்று'},
    'two': {'hi': 'दो', 'ben': 'দুই', 'mar': 'दोन', 'tel': 'రెండు', 'tam': 'இரண்டு'},
    'three': {'hi': 'तीन', 'ben': 'তিন', 'mar': 'तीन', 'tel': 'మూడు', 'tam': 'మూன்று'},
    'four': {'hi': 'चार', 'ben': 'চার', 'mar': 'चार', 'tel': 'నాలుగు', 'tam': 'நான்கு'},
    'five': {'hi': 'पाँच', 'ben': 'পাঁচ', 'mar': 'पाच', 'tel': 'ఐదు', 'tam': 'ஐந்து'},
    'seven': {'hi': 'सात', 'ben': 'সাত', 'mar': 'सात', 'tel': 'ఏడు', 'tam': 'ஏழு'},
    'nine': {'hi': 'नौ', 'ben': 'নয়', 'mar': 'नऊ', 'tel': 'తొమ్మిది', 'tam': 'ஒன்பது'},
    'ten': {'hi': 'दस', 'ben': 'দশ', 'mar': 'दहा', 'tel': 'పది', 'tam': 'பத்து'},
    'summer': {'hi': 'गर्मी', 'ben': 'গ্রীষ্ম', 'mar': 'उन्हाळा', 'tel': 'వేసవి', 'tam': 'கோடை'},
    'spring': {'hi': 'वसंत', 'ben': 'বসন্ত', 'mar': 'वसंत', 'tel': 'వసంతం', 'tam': 'வசந்தம்'},
    'gold': {'hi': 'सोना', 'ben': 'সোনা', 'mar': 'सोने', 'tel': 'బంగారం', 'tam': 'தங்கம்'},
    'ground': {'hi': 'मैदान', 'ben': 'মাঠ', 'mar': 'जमीन', 'tel': 'నేల', 'tam': 'தரை'},
    'left': {'hi': 'बायाँ', 'ben': 'বাম', 'mar': 'डावा', 'tel': 'ఎడమ', 'tam': 'இடது'},
    'defeat': {'hi': 'हार', 'ben': 'পরাজয়', 'mar': 'पराभव', 'tel': 'ఓటమి', 'tam': 'தோல்வி'},
    'middle': {'hi': 'मध्य', 'ben': 'মাঝ', 'mar': 'मध्य', 'tel': 'మధ్య', 'tam': 'நடு'},
    'flower': {'hi': 'फूल', 'ben': 'ফুল', 'mar': 'फूल', 'tel': 'పువ్వు', 'tam': 'மலர்'},
    'hand': {'hi': 'हाथ', 'ben': 'হাত', 'mar': 'हात', 'tel': 'చేయి', 'tam': 'கை'},
    'eye': {'hi': 'आँख', 'ben': 'চোখ', 'mar': 'डोळा', 'tel': 'కన్ను', 'tam': 'கண்'},
    'foot': {'hi': 'पैर', 'ben': 'পা', 'mar': 'पाय', 'tel': 'పాదం', 'tam': 'கால்'},
    'return': {'hi': 'वापसी', 'ben': 'প্রত্যাবর্তন', 'mar': 'परतावा', 'tel': 'తిరుగు', 'tam': 'மீண்டும்'}
}

files = {
    'ben': 'data/method1/ben_Deva_words_method1.csv',
    'mar': 'data/method1/mar_Deva_words_method1.csv',
    'tel': 'data/method1/tel_Deva_words_method1.csv',
    'tam': 'data/method1/tam_Deva_words_method1.csv'
}

for lang_code, filename in files.items():
    df = pd.read_csv(filename)
    l1_col = f"{lang_code}_word"
    
    # Clean punctuation and whitespace first
    df['hi_word'] = df['hi_word'].astype(str).str.replace(r'[।\.!?]', '', regex=True).str.strip()
    df[l1_col] = df[l1_col].astype(str).str.replace(r'[।\.!?]', '', regex=True).str.strip()
    
    # Strip Bengali clitics (-টি, -টা) and genitive (-র, -এর)
    if lang_code == 'ben':
        df[l1_col] = df[l1_col].str.replace(r'(টি|টা)$', '', regex=True)
        df[l1_col] = df[l1_col].str.replace(r'জন$', '', regex=True)
    
    # Apply ground-truth overrides
    for idx, row in df.iterrows():
        en = row['en_word']
        if en in CORRECTIONS:
            df.at[idx, 'hi_word'] = CORRECTIONS[en]['hi']
            df.at[idx, l1_col] = CORRECTIONS[en][lang_code]
            
    # Save the polished baseline
    out_path = f"data/method1/clean/{lang_code}_clean_final.csv"
    df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"Validated and saved: {out_path}")

Validated and saved: data/method1/clean/ben_clean_final.csv
Validated and saved: data/method1/clean/mar_clean_final.csv
Validated and saved: data/method1/clean/tel_clean_final.csv
Validated and saved: data/method1/clean/tam_clean_final.csv


An in-depth inspection of the uploaded CSV files reveals that **Tamil (`tam`) and Telugu (`tel`) are mostly clean**, but **Bengali (`ben`) and Marathi (`mar`) still contain critical errors**—including multi-word phrases, inflected case suffixes, and severe word-truncation bugs caused by aggressive regex stripping.

---

### 1. Severe Issues Requiring Fixes Before Tokenization

#### A. The Bengali Regex Over-Stripping Bug (Root Word Damage)

In the cleaning script, the regex rule `r'(টি|টা)$'` intended to strip the definite clitic (*the*) instead stripped genuine root letters from legitimate Bengali lemmas:

* **Row 4 (`soil`):** `মাটি` (*mati*) was stripped into **`মা`** (*ma* = mother). It must be restored to **`মাটি`**.
* **Row 67 (`pair`):** `জুটি` (*juti*) was stripped into **`জু`** (nonsense fragment). It must be restored to **`জুটি`**.
* **Row 110 (`attempt`):** `প্রচেষ্টা` (*prochesta*) was stripped into **`প্রচেষ্`** (broken incomplete conjunct). It must be restored to **`প্রচেষ্টা`**.

#### B. Lingering Multi-Word Phrases & Demonstratives

The paper's single-token evaluation strictly requires isolated, single-word lemmas. Several cells still contain compound phrases or demonstratives:

* **Bengali (`ben`):**
* Row 9 (`day`): **`সেই দিন`** (*that day*) $\rightarrow$ change to single word **`দিন`**.
* Row 81 (`kind`): **`সেই ধরনের`** (*that kind*) $\rightarrow$ change to **`প্রকার`** or **`ধরন`**.
* Row 90 (`delete`): **`মুছে ফেলা`** (2 words) $\rightarrow$ change to single word **`মোছা`** or **`অপসারণ`**.
* Row 121 (`person`): **`সেই ব্যক্তি`** (*that person*) $\rightarrow$ change to single word **`ব্যক্তি`**.


* **Marathi (`mar`):**
* Row 70 (`thousand`): **`एक हजार`** (2 words) $\rightarrow$ change to single word **`हजार`**.
* Row 81 (`kind`): **`असा प्रकार`** (2 words) $\rightarrow$ change to single word **`प्रकार`**.
* Row 97 (`meeting`): **`ही बैठक`** (2 words) $\rightarrow$ change to single word **`बैठक`**.
* Row 121 (`person`): **`त्या व्यक्तीची`** (2 words + genitive) $\rightarrow$ change to single word **`व्यक्ती`**.


* **Hindi (`hi` in all files):**
* Row 81 (`kind`): **`इस तरह का`** (3 words) $\rightarrow$ change to single word **`प्रकार`**.



#### C. Lingering Oblique / Genitive Case Endings

Some words were not caught by the dictionary overrides and retained possessive/locative case suffixes instead of nominative base forms:

* **Bengali (`ben`):**
* Row 21 (`sand`): **`বালির`** (*of sand*, genitive) $\rightarrow$ change to **`বালি`**.
* Row 87 (`face`): **`মুখের`** (*of face*, genitive) $\rightarrow$ change to **`মুখ`**.
* Row 101 (`generation`): **`প্রজন্মের`** (*of generation*, genitive) $\rightarrow$ change to **`প্রজন্ম`**.
* Row 117 (`child`): **`শিশুটির`** (*of the child*, genitive) $\rightarrow$ change to **`শিশু`**.
* Row 119 (`hair`): **`চুলের`** (*of hair*, genitive) $\rightarrow$ change to **`চুল`**.


* **Marathi (`mar`):**
* Row 79 (`friend`): **`मित्राला`** (*to the friend*, dative) $\rightarrow$ change to **`मित्र`**.
* Row 83 (`school`): **`शाळेची`** (*of the school*, genitive) $\rightarrow$ change to **`शाळा`**.
* Row 88 (`bridge`): **`पुलाचा`** (*of the bridge*, genitive) $\rightarrow$ change to **`पूल`**.
* Row 102 (`south`): **`दक्षिणेचा`** (*of the south*, genitive) $\rightarrow$ change to **`दक्षिण`**.
* Row 114 (`east`): **`पूर्वेकडे`** (*towards east*, directional) $\rightarrow$ change to **`पूर्व`**.
* Row 117 (`child`): **`मुलाची`** (*of the child*, genitive) $\rightarrow$ change to **`मुलगा`** or **`मूल`**.

In [4]:
import pandas as pd

# Definitive corrections for all remaining corrupted/multi-word/inflected tokens
PATCHES = {
    # Corrupted Bengali roots from regex over-stripping
    4:  {'ben': 'মাটি'},          # soil (was stripped to মা)
    67: {'ben': 'জুটি'},          # pair (was stripped to জু)
    110:{'ben': 'প্রচেষ্টা'},     # attempt (was stripped to প্রচেষ্)
    
    # Genitives and demonstratives in Bengali
    9:  {'ben': 'দিন'},            # day (was 'সেই দিন')
    21: {'ben': 'বালি'},           # sand (was 'বালির')
    81: {'ben': 'প্রকার', 'hi': 'प्रकार', 'mar': 'प्रकार', 'tel': 'రకం'}, # kind
    87: {'ben': 'মুখ'},            # face (was 'মুখের')
    90: {'ben': 'মোছা'},          # delete (was 'মুছে ফেলা')
    101:{'ben': 'প্রজন্ম'},        # generation (was 'প্রজন্মের')
    117:{'ben': 'শিশু'},          # child (was 'শিশুটির')
    119:{'ben': 'চুল'},           # hair (was 'চুলের')
    121:{'ben': 'ব্যক্তি', 'mar': 'व्यक्ती'}, # person (was 'সেই ব্যক্তি', 'त्या व्यक्तीची')
    
    # Marathi inflected suffixes and demonstratives
    70: {'mar': 'हजार'},          # thousand (was 'एक हजार')
    79: {'mar': 'मित्र'},          # friend (was 'मित्राला')
    83: {'mar': 'शाळा'},           # school (was 'शाळेची')
    88: {'mar': 'पूल'},           # bridge (was 'पुलाचा')
    97: {'mar': 'बैठक'},          # meeting (was 'ही बैठक')
    102:{'mar': 'दक्षिण'},         # south (was 'दक्षिणेचा')
    114:{'mar': 'पूर्व'},          # east (was 'पूर्वेकडे')
    117:{'mar': 'मूल'},            # child (was 'मुलाची')
}

files = {
    'ben': 'data/method1/clean/ben_clean_final.csv',
    'mar': 'data/method1/clean/mar_clean_final.csv',
    'tel': 'data/method1/clean/tel_clean_final.csv',
    'tam': 'data/method1/clean/tam_clean_final.csv'
}

for lang_code, fname in files.items():
    df = pd.read_csv(fname)
    l1_col = f"{lang_code}_word"
    
    for row_idx, patch_dict in PATCHES.items():
        if lang_code in patch_dict:
            df.at[row_idx, l1_col] = patch_dict[lang_code]
        if 'hi' in patch_dict:
            df.at[row_idx, 'hi_word'] = patch_dict['hi']
            
    # Save the repaired clean baseline
    out_file = f"data/method1/clean/final/{lang_code}.csv"
    df.to_csv(out_file, index=False, encoding='utf-8-sig')
    print(f"Patched and saved: {out_file}")

Patched and saved: data/method1/clean/final/ben.csv
Patched and saved: data/method1/clean/final/mar.csv
Patched and saved: data/method1/clean/final/tel.csv
Patched and saved: data/method1/clean/final/tam.csv


Moving on from here, we need sanitize the data at `data/method1/clean/final/` to account for following points:
1. The translation of a word shouldn't be the same word (otherwise it would be a repetition task)
2. The translation of a word should be a word which is also identified as a single token by the Sarvam Tokeniser.
3. Then we would also have to do some filtering depending on the Pivot Language (language used in Concept Space) we choose to investigate.